In [ ]:
import polars as pl
from pathlib import Path
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns


def add_label(label, data):
    return {x: f"{x}{label}" for x in data}

In [ ]:
# Assuming either all *_compressed.tar.zst files or all_groups_lean.tar.zst file is/are decompressed
CWD = Path().resolve()
BASE = CWD.parent

print(f"Current working directory (where figure will be saved): {CWD}")
print(f"Folder where groups (data) folder should be: {BASE}")

In [ ]:
groups = [
    "fungi_mit",
    "metazoans_mit",
    "green_algae_mit",
    "green_algae_plt",
    "plants_mit",
    "plants_plt",
    "protists_mit",
    "protists_plt",
]

label_map = {
    "fungi_mit": "Fungi (mitochondria)",
    "green_algae_mit": "Green algae (mitochondria)",
    "metazoans_mit": "Metazoans (mitochondria)",
    "plants_mit": "Plants (mitochondria)",
    "protists_mit": "Protists (mitochondria)",
    "green_algae_plt": "Green algae (plastid)",
    "plants_plt": "Plants (plastid)",
    "protists_plt": "Protists (plastid)",
}

polarity_2bin = {
    "++": "same",
    "--": "same",
    "+-": "opp",
    "-+": "opp",
}
label_2bin = set(polarity_2bin.values())

COLOR_pct = "#009E73"  # teal green
COLOR_BORDER = "#F0E442"  # yellow

In [ ]:
s1_data = []
for g in groups:
    tsv_path = BASE / g / f"{g}.tsv"
    brms_path = BASE / g / "brms_3bin" / "filtered_3bin.tsv"

    tsv = pl.read_csv(tsv_path, separator="\t")
    brms = pl.read_csv(brms_path, separator="\t")
    polarity = dict(brms["polarity_bin"].value_counts().iter_rows())

    s1_data.append(
        {
            "Taxonomic group and organelle type": label_map[g],
            "Total Genomes": tsv["AN"].n_unique(),
            "Total IGRs": brms.height,
            "Unique Taxa": tsv["ncbi_taxid"].n_unique(),
            "Unique Taxa in Tree": brms["taxon_tree"].n_unique(),
            "Same polarity IGRs": polarity.get("same", 0),
            "Convergent polarity IGRs": polarity.get("convergent", 0),
            "Divergent polarity IGRs": polarity.get("divergent", 0),
        }
    )

s1 = pl.DataFrame(s1_data)
s1.write_csv(BASE / "code" / "supplemental_table1.tsv", separator="\t")
print(s1)

In [ ]:
data_2bin = {}

for g in groups:
    print(f"Group: {g}")
    tsv = pl.read_csv(BASE / g / f"{g}.tsv", separator="\t")
    igs = pl.read_csv(BASE / g / "summary_igs_intergenic.tsv", separator="\t")

    igs = igs.with_columns(
        pl.col("Polarity").replace_strict(polarity_2bin).alias("polarity_2bin")
    )

    counts = igs.group_by(["AN", "polarity_2bin"]).agg(
        pl.col("Length").count().alias("count"),
    )
    wide_counts = counts.pivot(values="count", index="AN", on="polarity_2bin")
    wide_counts = wide_counts.rename(add_label("_count", label_2bin))

    igs_sizes = igs.group_by("AN").agg(pl.col("Length").sum().alias("total_igs_size"))

    wide = (
        wide_counts.join(igs_sizes, on="AN", how="left", validate="1:1")
        .join(tsv.select(["AN", "Genome_length"]), on="AN", how="left", validate="1:1")
        .with_columns(
            (pl.col("total_igs_size") / pl.col("Genome_length") * 100).alias(
                "noncoding_pct"
            ),
            (
                pl.col("opp_count") / (pl.col("opp_count") + pl.col("same_count")) * 100
            ).alias("count_opp_pct"),
        )
    )

    data_2bin[g] = wide

In [ ]:
fig, axs = plt.subplots(4, 2, figsize=(15, 20))
axs = axs.flatten()

for i, g in enumerate(groups):
    wide = data_2bin[g]
    x = wide["count_opp_pct"].to_list()
    y = wide["noncoding_pct"].to_list()
    n = len(x)

    # scatter + regression + CI
    sns.regplot(
        x=x,
        y=y,
        ax=axs[i],
        scatter_kws={"color": COLOR_pct, "alpha": 0.6, "s": 15},
        line_kws={"color": "black", "lw": 1.5},
        ci=95,
    )

    # stats annotation
    res = stats.linregress(x, y)
    rho, p_s = stats.spearmanr(x, y)
    axs[i].text(
        0.95,
        0.95,
        f"Pearson  $r$ = {res.rvalue:+.3f} ($p$ = {res.pvalue:.1g})\n"
        f"Spearman $\\rho$ = {rho:+.3f} ($p$ = {p_s:.1g})\n"
        f"$n$ = {n}",
        transform=axs[i].transAxes,
        horizontalalignment="right",
        verticalalignment="top",
        fontsize=10,
        family="monospace",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8, edgecolor="none"),
    )

    axs[i].set_xlabel("Proportion of Opposite Polarity Gene Pairs (%)", fontsize=13)
    axs[i].set_ylabel("Proportion of Noncoding DNA (%)", fontsize=13)
    axs[i].set_title(label_map[g], fontsize=14)
    axs[i].tick_params(axis="both", labelsize=13)

plt.suptitle(
    "Per-genome percentage of opposite polarity gene pairs vs. percentage of noncoding DNA",
    fontsize=16,
    fontweight="bold",
    y=1.001,
)
plt.tight_layout()
plt.savefig("supplemental_figure1.png", dpi=300, bbox_inches="tight")
plt.show()